
# Prediction Of Heart Disease Using Ensemble<br>Learning & Ant Colony Optimization

### 1. Importing Libraries
Importing all the necessary libraries.

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import re
import random
import math
import time
from datetime import timedelta
from sklearn.model_selection import train_test_split
from sklearn import tree
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import BaggingClassifier, ExtraTreesClassifier
from sklearn.metrics import accuracy_score
from scipy import spatial
from scipy.spatial.distance import cosine
from sklearn.preprocessing import normalize
from sklearn import svm
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

### 2. Loading Dataset
Loading the required Dataset as Dataframe.

In [16]:
data1 = pd.read_csv("heart project.csv")
data1


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1020,59,1,1,140,221,0,1,164,1,0.0,2,0,2,1
1021,60,1,0,125,258,0,0,141,1,2.8,1,1,3,0
1022,47,1,0,110,275,0,0,118,1,1.0,1,1,2,0
1023,50,0,0,110,254,0,0,159,0,0.0,2,0,2,1


### 3. Data Cleaning and Preprocessing
Replacing the column names of the dataset with their actual names.<br>
Converting the numeric features to categorical features.

In [17]:
data1.columns = ['age','sex','chest pain type','resting blood pressure',
                 'cholesterol','fasting blood sugar','rest ecg',
                 'max heart rate achieved','exercise induced angina','oldpeak',
                 'st slope','no. of vessels colored','thalassemia','target']
data1['chest pain type'][data1['chest pain type'] == 0] = 'typical angina'
data1['chest pain type'][data1['chest pain type'] == 1] = 'atypical angina'
data1['chest pain type'][data1['chest pain type'] == 2] = 'non-anginal pain'
data1['chest pain type'][data1['chest pain type'] == 3] = 'asymptomatic'

data1['rest ecg'][data1['rest ecg'] == 0] = 'normal'
data1['rest ecg'][data1['rest ecg'] == 1] = 'ST-T wave abnormality'
data1['rest ecg'][data1['rest ecg'] == 2] = 'left ventricular hypertrophy'

data1['st slope'][data1['st slope'] == 0] = 'normal'
data1['st slope'][data1['st slope'] == 1] = 'upsloping'
data1['st slope'][data1['st slope'] == 2] = 'flat'
data1['st slope'][data1['st slope'] == 3] = 'downsloping'

data1['thalassemia'][data1['thalassemia'] == 0] = 'no thalassemia'
data1['thalassemia'][data1['thalassemia'] == 1] = 'normal thalassemia'
data1['thalassemia'][data1['thalassemia'] == 2] = 'fixed defect thalassemia'
data1['thalassemia'][data1['thalassemia'] == 3] = 'reversible defect thalassemia'

data1['fasting blood sugar'][data1['fasting blood sugar'] == 0] = 'Fasting Blood Sugar <= 120'
data1['fasting blood sugar'][data1['fasting blood sugar'] == 1] = 'Fasting Blood Sugar > 120'

data1['exercise induced angina'][data1['exercise induced angina'] == 0] = 'no exercise induced angina'
data1['exercise induced angina'][data1['exercise induced angina'] == 1] = 'exercise induced angina'

data1['target'][data1['target'] == 0] = 'healthy'
data1['target'][data1['target'] == 1] = 'heart disease'

data1['sex'][data1['sex'] == 0] = 'female'
data1['sex'][data1['sex'] == 1] = 'male'

data1

,age,sex,chest pain type,resting blood pressure,cholesterol,fasting blood sugar,rest ecg,max heart rate achieved,exercise induced angina,oldpeak,st slope,no. of vessels colored,thalassemia,target
0,52,male,typical angina,125,212,Fasting Blood Sugar <= 120,ST-T wave abnormality,168,no exercise induced angina,1.0,flat,2,reversible defect thalassemia,healthy
1,53,male,typical angina,140,203,Fasting Blood Sugar > 120,normal,155,exercise induced angina,3.1,normal,0,reversible defect thalassemia,healthy
2,70,male,typical angina,145,174,Fasting Blood Sugar <= 120,ST-T wave abnormality,125,exercise induced angina,2.6,normal,0,reversible defect thalassemia,healthy
3,61,male,typical angina,148,203,Fasting Blood Sugar <= 120,ST-T wave abnormality,161,no exercise induced angina,0.0,flat,1,reversible defect thalassemia,healthy
4,62,female,typical angina,138,294,Fasting Blood Sugar > 120,ST-T wave abnormality,106,no exercise induced angina,1.9,upsloping,3,fixed defect thalassemia,healthy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1020,59,male,atypical angina,140,221,Fasting Blood Sugar <= 120,ST-T wave abnormality,164,exercise induced angina,0.0,flat,0,fixed defect thalassemia,heart disease
1021,60,male,typical angina,125,258,Fasting Blood Sugar <= 120,normal,141,exercise induced angina,2.8,upsloping,1,reversible defect thalassemia,healthy
1022,47,male,typical angina,110,275,Fasting Blood Sugar <= 120,normal,118,exercise induced angina,1.0,upsloping,1,fixed defect thalassemia,healthy
1023,50,female,typical angina,110,254,Fasting Blood Sugar <= 120,normal,159,no exercise induced angina,0.0,flat,0,fixed defect thalassemia,heart disease


Checking missing entries in the dataset columnwise.

In [18]:
data1.isna().sum()

age                        0
sex                        0
chest pain type            0
resting blood pressure     0
cholesterol                0
fasting blood sugar        0
rest ecg                   0
max heart rate achieved    0
exercise induced angina    0
oldpeak                    0
st slope                   0
no. of vessels colored     0
thalassemia                0
target                     0
dtype: int64

#### This concludes that the dataset has no null values

Describing numeric features of the dataset.

In [19]:
data1.describe(include =[np.number])

,age,resting blood pressure,cholesterol,max heart rate achieved,oldpeak,no. of vessels colored
count,1025.000000,1025.000000,1025.00000,1025.000000,1025.000000,1025.000000
mean,54.434146,131.611707,246.00000,149.114146,1.071512,0.754146
std,9.072290,17.516718,51.59251,23.005724,1.175053,1.030798
min,29.000000,94.000000,126.00000,71.000000,0.000000,0.000000
25%,48.000000,120.000000,211.00000,132.000000,0.000000,0.000000
50%,56.000000,130.000000,240.00000,152.000000,0.800000,0.000000
75%,61.000000,140.000000,275.00000,166.000000,1.800000,1.000000
max,77.000000,200.000000,564.00000,202.000000,6.200000,4.000000


Describing categorical features of the dataset.

In [20]:
data1.describe(include =[object])

,sex,chest pain type,fasting blood sugar,rest ecg,exercise induced angina,st slope,thalassemia,target
count,1025,1025,1025,1025,1025,1025,1025,1025
unique,2,4,2,3,2,3,4,2
top,male,typical angina,Fasting Blood Sugar <= 120,ST-T wave abnormality,no exercise induced angina,upsloping,fixed defect thalassemia,heart disease
freq,713,497,872,513,680,482,544,526


### 5. ACO Feature Selection and Model Building


Applying Hybrid model(Random forest and Gradient boosting) and calculating Accuracy.

In [21]:
data1 = pd.read_csv("heart project.csv")
data1

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1020,59,1,1,140,221,0,1,164,1,0.0,2,0,2,1
1021,60,1,0,125,258,0,0,141,1,2.8,1,1,3,0
1022,47,1,0,110,275,0,0,118,1,1.0,1,1,2,0
1023,50,0,0,110,254,0,0,159,0,0.0,2,0,2,1


In [22]:
data_columns = data1.drop(['target'], axis=1)
label = data1['target'].values
from sklearn import preprocessing
normalized_data = preprocessing.normalize(data_columns)
for i in range(0, 303):
    data_columns.loc[i, ('age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope',
                         'ca', 'thal')] = normalized_data[i]

X_train, X_test, y_train, y_test = train_test_split(data_columns, label, test_size=0.20, random_state=42)

# Train Random Forest and Gradient Boosting models
rf_clf = RandomForestClassifier(n_estimators=10, random_state=42)
gb_clf = GradientBoostingClassifier(random_state=42)

rf_clf.fit(X_train, y_train)
gb_clf.fit(X_train, y_train)

# Get predictions from both models
rf_preds = rf_clf.predict_proba(X_test)[:, 1]
gb_preds = gb_clf.predict_proba(X_test)[:, 1]

# Combine predictions as features for the meta-classifier
stacked_features = np.column_stack((rf_preds, gb_preds))

# Train a meta-classifier (e.g., Logistic Regression)
meta_clf = LogisticRegression()
meta_clf.fit(stacked_features, y_test)

# Make final predictions
final_preds = meta_clf.predict(stacked_features)

# Evaluate the combined model
accuracy = accuracy_score(y_test, final_preds) * 100
precision = precision_score(y_test, final_preds, average='binary')
recall = recall_score(y_test, final_preds, average='binary')
f1 = f1_score(y_test, final_preds, average='binary')

print("Accuracy for Combined Model (Random Forest + Gradient Boosting):", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Accuracy for Combined Model (Random Forest + Gradient Boosting): 94.14634146341463
Precision: 0.941747572815534
Recall: 0.941747572815534
F1 Score: 0.941747572815534


Applying Hybrid model(Random forest and Gradient boosting) with ACO and calculating Accuracy.

In [23]:
X = data1.drop(['target'], axis=1)
y = data1['target']

# Normalize data
X_normalized = normalize(X)

# Define ACO for feature selection
class ACO:
    def __init__(self, num_features, num_ants, num_iterations, evaporation_rate, alpha, beta):
        self.num_features = num_features
        self.num_ants = num_ants
        self.num_iterations = num_iterations
        self.evaporation_rate = evaporation_rate
        self.alpha = alpha
        self.beta = beta
        self.pheromone = np.ones((num_features, num_features))

    def run(self, X, y):
        best_features = None
        best_accuracy = 0

        for _ in range(self.num_iterations):
            for ant in range(self.num_ants):
                selected_features = self.select_features()
                accuracy = self.evaluate_features(X[:, selected_features], y)

                if accuracy > best_accuracy:
                    best_accuracy = accuracy
                    best_features = selected_features

                self.update_pheromone(selected_features, accuracy)

        return best_features

    def select_features(self):
        selected_features = []
        for i in range(self.num_features):
            probabilities = self.pheromone[i] ** self.alpha
            probabilities /= probabilities.sum()
            selected_features.append(np.random.choice(range(self.num_features), p=probabilities))
        return list(set(selected_features))

    def evaluate_features(self, X, y):
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        model = RandomForestClassifier()
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        return accuracy_score(y_test, y_pred)

    def update_pheromone(self, selected_features, accuracy):
        for i in selected_features:
            for j in selected_features:
                self.pheromone[i, j] *= (1 - self.evaporation_rate)
                self.pheromone[i, j] += accuracy

# Run ACO for feature selection
aco = ACO(num_features=X.shape[1], num_ants=10, num_iterations=20, evaporation_rate=0.1, alpha=1, beta=2)
selected_features = aco.run(X_normalized, y)

# Train Random Forest and Gradient Boosting on selected features
X_selected = X.iloc[:, selected_features]
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)

rf_clf = RandomForestClassifier(n_estimators=10, random_state=42)
gb_clf = GradientBoostingClassifier(random_state=42)

rf_clf.fit(X_train, y_train)
gb_clf.fit(X_train, y_train)

# Combine predictions using Logistic Regression
rf_preds = rf_clf.predict_proba(X_test)[:, 1]
gb_preds = gb_clf.predict_proba(X_test)[:, 1]
stacked_features = np.column_stack((rf_preds, gb_preds))

meta_clf = LogisticRegression()
meta_clf.fit(stacked_features, y_test)
final_preds = meta_clf.predict(stacked_features)

# Evaluate the combined model
accuracy = accuracy_score(y_test, final_preds) * 100
precision = precision_score(y_test, final_preds)
recall = recall_score(y_test, final_preds)
f1 = f1_score(y_test, final_preds)

print("combined model of random forest and Gradient boosting with Ant colony optimizer")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

combined model of random forest and Gradient boosting with Ant colony optimizer
Accuracy: 98.53658536585365
Precision: 1.0
Recall: 0.970873786407767
F1 Score: 0.9852216748768473
